# W14C1 Lab: An Agent That Uses Tools

Run every cell from the top. **Everything already works.**

The reasoning here is scripted rather than sampled, so the loop is the
same every time you run it and you can read exactly what happened. A real
model drops into the same loop; it is just less predictable.

Today you will:

1. Watch a language model fail at something a calculator finds trivial.
2. Give it tools and run the ReAct loop, one step at a time.
3. Break the agent by writing a bad tool description.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
import pandas as pd

# Three tools. Each is an ordinary Python function with a description.
def calculator(expression):
    """Evaluate an arithmetic expression, e.g. '12 * 7'."""
    return str(eval(expression, {"__builtins__": {}}, {}))

def stock(symbol):
    """Look up a share price by ticker symbol, e.g. 'ACME'."""
    prices = {"ACME": 42.50, "GLOBEX": 118.20, "INITECH": 7.05}
    return str(prices.get(symbol.upper(), "unknown symbol"))

def staff_count(department):
    """How many people work in a department, e.g. 'sales'."""
    counts = {"sales": 24, "engineering": 61, "support": 18}
    return str(counts.get(department.lower(), "unknown department"))

TOOLS = {"calculator": calculator, "stock": stock, "staff_count": staff_count}

for name, fn in TOOLS.items():
    print(f"   {name:<12} {fn.__doc__.splitlines()[0]}")

## Part 1. Why a model needs tools at all

A language model predicts text. It does not compute, look things up, or
know today's price of anything. Those are exactly the jobs it is worst at.

In [ ]:
# GIVEN. What the model has to do without help.
QUESTIONS = [
    "What is 4173 times 219?",
    "What is the ACME share price?",
    "How many people work in engineering?",
]
print("Without tools a model must produce these from memory alone:")
for q in QUESTIONS:
    print("   ", q)
print()
print("The first has one right answer it was never taught.")
print("The second changes every day.")
print("The third is in a database it has never seen.")
print()
print("correct answers:", calculator("4173 * 219"), stock("ACME"), staff_count("engineering"))

## Part 2. The ReAct loop

Thought, Action, Observation, repeated. The model reasons about what it
needs, calls a tool, reads the result, and goes round again until it can
answer. Everything an agent does is this loop.

In [ ]:
# GIVEN. A scripted reasoner, so the loop is identical on every run.
def reasoner(question, history):
    """Stands in for the model. Decides the next Action from what it has seen."""
    if history:                                   # already have an observation
        return {"thought": "I have what I need.", "action": None,
                "answer": history[-1]["observation"]}
    if re.search(r"\d+\s*[-+*/x]\s*\d+|times|plus|divided", question):
        expr = re.sub(r"[^\d+\-*/(). ]", " ", question.replace("times", "*"))
        return {"thought": "This is arithmetic, use the calculator.",
                "action": ("calculator", expr.strip()), "answer": None}
    if "share price" in question or "stock" in question:
        symbol = re.search(r"\b([A-Z]{3,})\b", question)
        return {"thought": "I need a live price, use the stock tool.",
                "action": ("stock", symbol.group(1) if symbol else "ACME"), "answer": None}
    if "work in" in question or "people" in question:
        dept = question.rstrip("?").split()[-1]
        return {"thought": "This is a database lookup, use staff_count.",
                "action": ("staff_count", dept), "answer": None}
    return {"thought": "No tool fits.", "action": None, "answer": "I don't know."}

def run_agent(question, think=reasoner, max_steps=4, verbose=True):
    """`think` is the policy. Swap it out to change how the agent decides."""
    history = []
    for step in range(max_steps):
        decision = think(question, history)
        if verbose:
            print(f"   Thought: {decision['thought']}")
        if decision["action"] is None:
            if verbose:
                print(f"   Answer : {decision['answer']}")
            return decision["answer"], history
        name, arg = decision["action"]
        observation = TOOLS[name](arg)
        history.append({"action": name, "arg": arg, "observation": observation})
        if verbose:
            print(f"   Action : {name}({arg!r})")
            print(f"   Observe: {observation}")
    return None, history

for q in QUESTIONS:
    print(f"Q: {q}")
    run_agent(q)
    print()

In [ ]:
# ================== YOUR TURN 1 ==================
# Add a fourth tool and a question that needs it.
#
# Write the function, add it to TOOLS, then add one branch to the
# reasoner so it knows when to reach for it.
#
# Expected: your question routes to your tool and comes back with the right
#           answer. Notice how much of an agent is plumbing: the intelligence is
#           in choosing the tool, and everything else is ordinary code.
# ===============================================
def weather(city):
    """Today's weather for a city, e.g. 'san antonio'."""
    forecasts = {"san antonio": "32C and sunny", "london": "14C and raining"}
    return forecasts.get(city.lower(), "no data")

TOOLS["weather"] = weather          # <-- registered

def reasoner_v2(question, history):
    if history:
        return {"thought": "I have what I need.", "action": None,
                "answer": history[-1]["observation"]}

    # <-- add your branch here, for example:
    #     if "weather" in question.lower():
    #         city = question.rstrip("?").split(" in ")[-1]
    #         return {"thought": "Use the weather tool.",
    #                 "action": ("weather", city), "answer": None}

    return reasoner(question, history)      # fall through to the original

MY_QUESTION = "What is the weather in London?"
print(f"Q: {MY_QUESTION}")
run_agent(MY_QUESTION, think=reasoner_v2)

## Part 3. Tool descriptions are prompts

A real agent picks its tool by reading the descriptions you wrote. A vague
or wrong description is a bug, and it fails in a way that looks like the
model being stupid.

In [ ]:
# GIVEN. The descriptions the model would actually see.
for name, fn in TOOLS.items():
    print(f"   {name:<12} {fn.__doc__.splitlines()[0]}")
print()
print("Read them as a stranger would. Which two could be confused?")
print("'stock' takes a ticker symbol. Nothing here says what happens if you")
print("pass a company NAME instead, and nothing says the data is fake.")

In [ ]:
# ================== YOUR TURN 2 ==================
# Rewrite one description so it is unambiguous: say what the argument
# looks like, what comes back, and what happens when it fails.
#
# Expected: a good description names the argument format, the return type and the
#           failure mode. That is the entire interface between the model and your
#           code, and it is written in English, which is why agent debugging so
#           often turns out to be editing a docstring.
# ===============================================
def stock_v2(symbol):
    """Look up a share price by ticker symbol, e.g. 'ACME'."""   # <-- rewrite this
    prices = {"ACME": 42.50, "GLOBEX": 118.20, "INITECH": 7.05}
    return str(prices.get(symbol.upper(), "unknown symbol"))

print("old:", stock.__doc__.strip())
print("new:", stock_v2.__doc__.strip())
print()
print("test the failure path:", stock_v2("Apple Inc"))

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   A branch such as:
#       if "weather" in question:
#           city = question.rstrip("?").split(" in ")[-1]
#           return {"thought": "Use the weather tool.",
#                   "action": ("weather", city), "answer": None}
#   Most of an agent is this: routing, argument extraction, and error handling.
#
# YOUR TURN 2
#   For example: "Look up today's share price for a ticker symbol. Argument: a
#   3 to 6 letter uppercase symbol such as 'ACME', NOT a company name. Returns a
#   price as a string, or 'unknown symbol' if the ticker is not recognised."
#   The model has nothing but this text to go on.